# PneumoType: what kind of pneumonia is this X-ray?

**User:** a doctor at a busy children's clinic who has to decide fast whether a child with a cough needs antibiotics.
**Input → output:** one chest X-ray → **Normal / Bacterial pneumonia / Viral pneumonia**, a confidence score, a heatmap, and **"Unsure, refer"** when the model isn't confident.

Why the *type* matters: bacterial pneumonia needs antibiotics; viral usually doesn't. The dangerous mistake is calling a **bacterial** case **viral or normal** (the child misses antibiotics).

**Everything is trained from scratch:** our own convolutional neural network, random starting weights, no pretrained model.

> **This project is a prototype and is not medical advice.**

**Setup on Kaggle:** New Notebook → *Add Input* → search **"Chest X-Ray Images (Pneumonia)"** (Paul Mooney) → add it. Settings → Accelerator → **GPU T4**. Turn Internet **on** (needed for `pip install`). Then Run All.

In [ ]:
!pip install -q gradio

## 1. Settings
`MODE="3class"` predicts normal / bacterial / viral. `MODE="binary"` predicts just normal vs pneumonia. Our model is a CNN we designed ourselves and train **from scratch** (no pretrained weights).

In [ ]:
import os, re, glob, random, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
IMG_SIZE = 224
BATCH = 32
EPOCHS = 25         # small CNN: ~30-60 s per epoch on a T4
LR = 1e-3
MODE = "3class"     # "3class" (normal/bacterial/viral) or "binary" (normal/pneumonia)
DATA_ROOT = "/kaggle/input"
CLASSES = ["normal", "bacterial", "viral"] if MODE == "3class" else ["normal", "pneumonia"]
K = len(CLASSES)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 2. Load file list and labels
Labels come from the folder (NORMAL / PNEUMONIA) and, for pneumonia, from the filename (`person12_bacteria_45.jpeg` or `person3_virus_7.jpeg`).
We also pull out a **patient ID** so the same child never appears in both training and test (otherwise the score is inflated).

In [ ]:
paths = [p for p in glob.glob(f"{DATA_ROOT}/**/*.jpeg", recursive=True) if "__MACOSX" not in p]

def label_of(p):
    folder = os.path.basename(os.path.dirname(p)).upper()
    name = os.path.basename(p).lower()
    if folder == "NORMAL":
        return "normal"
    if folder == "PNEUMONIA":
        if MODE == "binary": return "pneumonia"
        if "bacteria" in name: return "bacterial"
        if "virus" in name: return "viral"
    return None

def patient_of(p):
    name = os.path.basename(p)
    m = re.match(r"(person\d+)_", name, re.I)
    if m: return m.group(1).lower()
    m = re.match(r"(.*?IM-\d+)-\d+", name, re.I)
    if m: return m.group(1).upper()
    return os.path.splitext(name)[0]

df = pd.DataFrame({"path": paths})
df["label"] = df.path.map(label_of)
df = df.dropna(subset=["label"])
df["fname"] = df.path.map(os.path.basename)
df = df.drop_duplicates(subset=["label", "fname"])   # the Kaggle copy has a duplicated nested folder
df["patient"] = df.path.map(patient_of)
df["y"] = df.label.map({c: i for i, c in enumerate(CLASSES)})

print(len(df), "images,", df.patient.nunique(), "patients")
print(df.label.value_counts())
assert len(df) > 1000, "Images not found: check the dataset is attached and DATA_ROOT is right"

## 3. Split by patient: 70% train / 15% validation / 15% test
The test set is locked away until the very end. Validation is used to pick the best epoch and the thresholds.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def group_split(d, n_splits, seed):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    a, b = next(sgkf.split(d, d.y, groups=d.patient))
    return d.iloc[a].reset_index(drop=True), d.iloc[b].reset_index(drop=True)

trainval, test_df = group_split(df, 7, SEED)          # ~15% test
train_df, val_df  = group_split(trainval, 6, SEED)    # ~15% val

assert not set(train_df.patient) & set(test_df.patient)
assert not set(val_df.patient) & set(test_df.patient)
for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(n, len(d), d.label.value_counts().to_dict())

## 4. Baseline: always guess the most common class
Our model has to beat this. Accuracy alone can look good here, so we also report **macro recall** (the average of per-class recall), which a lazy model can't game.

In [ ]:
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

def report(name, y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    rec = recall_score(y_true, y_pred, labels=list(range(K)), average=None, zero_division=0)
    yt_p = (y_true > 0).astype(int); yp_p = (y_pred > 0).astype(int)
    row = {"model": name, "accuracy": accuracy_score(y_true, y_pred), "macro_recall": rec.mean()}
    for c, r in zip(CLASSES, rec): row[f"recall_{c}"] = r
    row["pneumonia_recall(any type)"] = recall_score(yt_p, yp_p, zero_division=0)
    row["false_alarms(normal called pneumonia)"] = int(((yt_p == 0) & (yp_p == 1)).sum())
    row["missed_pneumonia(called normal)"] = int(((yt_p == 1) & (yp_p == 0)).sum())
    row[f"DANGER {CLASSES[1]} missed"] = int(((y_true == 1) & (y_pred != 1)).sum())
    return row

results = []
majority = train_df.y.mode()[0]
results.append(report(f"Baseline: always '{CLASSES[majority]}'", test_df.y, [majority] * len(test_df)))
pd.DataFrame(results)

## 5. Images → tensors
Light augmentation for training only (small rotations/zoom/brightness). No left-right flips: the heart is on one side, so flipping makes unrealistic X-rays.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])

def to_model_input(x):
    # x: [B,1,H,W] in [0,1]
    return (x - 0.5) / 0.25                # simple centering; 1 grayscale channel

class XRays(Dataset):
    def __init__(self, d, tf): self.d, self.tf = d.reset_index(drop=True), tf
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d.iloc[i]
        img = Image.open(r.path).convert("L")
        return self.tf(img), int(r.y)

nw = 2
train_dl = DataLoader(XRays(train_df, train_tf), batch_size=BATCH, shuffle=True, num_workers=nw)
val_dl   = DataLoader(XRays(val_df, eval_tf), batch_size=BATCH, num_workers=nw)
test_dl  = DataLoader(XRays(test_df, eval_tf), batch_size=BATCH, num_workers=nw)

## 6. Model
Our own CNN, trained from random weights. Five blocks of *convolution → batch norm → ReLU → max-pool*, doubling the number of filters each block (32 → 512), then global average pooling, dropout and a final linear layer.
- **Convolutions** learn small patterns (edges, then textures, then shapes like hazy patches in the lungs).
- **Max-pooling** shrinks the image so later layers see bigger regions.
- **Batch norm** keeps training stable; **dropout** reduces overfitting.
About 4.7M parameters, small enough to train in minutes on a free GPU.

In [ ]:
def block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.MaxPool2d(2))

class OurCNN(nn.Module):
    def __init__(self):
        super().__init__()
        chans = [1, 32, 64, 128, 256, 512]
        self.features = nn.Sequential(*[block(a, b) for a, b in zip(chans[:-1], chans[1:])])

class PneumoType(nn.Module):
    def __init__(self, n_out=None):
        super().__init__()
        self.backbone = OurCNN()
        self.drop = nn.Dropout(0.3)
        self.head = nn.Linear(512, n_out or K)
    def forward(self, x):
        f = self.backbone.features(to_model_input(x))
        f = F.adaptive_avg_pool2d(F.relu(f), 1).flatten(1)
        return self.head(self.drop(f))
    def embed(self, x):
        f = self.backbone.features(to_model_input(x))
        return F.adaptive_avg_pool2d(F.relu(f), 1).flatten(1)

model = PneumoType().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M params")

## 7. A second baseline: logistic regression on raw pixels
A simple, non-deep model: shrink each X-ray to 32×32 pixels and fit a logistic regression. If our CNN can't clearly beat this, it isn't learning much.

In [ ]:
from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing import StandardScaler
def pixels(d):
    return np.stack([np.asarray(Image.open(p).convert("L").resize((32, 32)), dtype=np.float32).ravel() / 255 for p in d.path])

sc = StandardScaler().fit(pixels(train_df))
lr = LogisticRegression(max_iter=3000, class_weight="balanced", C=0.1).fit(sc.transform(pixels(train_df)), train_df.y)
results.append(report("Baseline: logistic regression on 32x32 pixels", test_df.y, lr.predict(sc.transform(pixels(test_df)))))
pd.DataFrame(results)

## 8. Train our CNN from scratch
Class weights make the rarer classes count more, so the model doesn't just ignore it. We keep the epoch with the best **macro recall on validation** (never on test).

In [ ]:
counts = train_df.y.value_counts().sort_index().values
class_w = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float32, device=device)

@torch.no_grad()
def predict_probs(dl, m=None):
    m = m or model; m.eval(); P, Y = [], []
    for x, y in dl:
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            P.append(F.softmax(m(x.to(device)).float(), 1).cpu().numpy())
        Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

def train_model(m, dl, path, epochs=EPOCHS):
    criterion = nn.CrossEntropyLoss(weight=class_w)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
    best = -1
    for ep in range(epochs):
        m.train(); t0 = time.time(); tot = 0
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                loss = criterion(m(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(y)
        sched.step()
        pv, yv = predict_probs(val_dl, m)
        mr = recall_score(yv, pv.argmax(1), average="macro")
        print(f"epoch {ep+1}: train loss {tot/len(dl.dataset):.3f} | val macro recall {mr:.3f} | {time.time()-t0:.0f}s")
        if mr > best:
            best = mr; torch.save(m.state_dict(), path)
    m.load_state_dict(torch.load(path))
    print("best val macro recall:", round(best, 3))
    return best

train_model(model, train_dl, "best.pt")

## 9. Results on the held-out test set

In [ ]:
p_test, y_test = predict_probs(test_dl)
pred_test = p_test.argmax(1)
results.append(report("Our CNN (from scratch)", y_test, pred_test))
display(pd.DataFrame(results).round(3))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test, labels=list(range(K))),
                       display_labels=CLASSES).plot(cmap="Blues")
plt.title("Test set confusion matrix"); plt.show()

## 10. Safety tuning: make the dangerous mistake rarer
We add a small bonus to the score of class 1 (**bacterial** in 3-class mode, **pneumonia** in binary mode), choosing the smallest bonus that keeps its **recall ≥ 95% on validation**, then check what it costs on test.
We also add an **"Unsure, refer"** band: if the top probability is below a cutoff, the tool doesn't decide.

In [ ]:
def adjusted_pred(p, bonus):
    q = p.copy(); q[:, 1] = q[:, 1] + bonus
    return q.argmax(1)

p_val, y_val = predict_probs(val_dl)
TARGET = 0.95
chosen = 0.0
for b in np.arange(0, 0.6, 0.01):
    if recall_score(y_val, adjusted_pred(p_val, b), labels=[1], average=None)[0] >= TARGET:
        chosen = b; break
print(f"{CLASSES[1]} bonus chosen on validation:", round(chosen, 2))

results.append(report(f"Our CNN + safety bonus {chosen:.2f}", y_test, adjusted_pred(p_test, chosen)))
display(pd.DataFrame(results).round(3))

# Unsure band
for cut in [0.5, 0.6, 0.7, 0.8]:
    conf = p_test.max(1); keep = conf >= cut
    acc = accuracy_score(y_test[keep], adjusted_pred(p_test, chosen)[keep]) if keep.any() else float("nan")
    print(f"cutoff {cut}: decides on {keep.mean():.0%} of X-rays, accuracy on those {acc:.3f}, refers {(~keep).sum()}")

## 11. Where does it fail?
The most confident **wrong** answers on the test set. Look at them and write down *why* you think the model got each one wrong (bad image quality, mild disease, devices or labels on the image, etc.).

In [ ]:
wrong = np.where(pred_test != y_test)[0]
wrong = wrong[np.argsort(-p_test[wrong].max(1))][:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, i in zip(axes.flat, wrong):
    ax.imshow(Image.open(test_df.path[i]).convert("L"), cmap="gray")
    ax.set_title(f"true {CLASSES[y_test[i]]} | pred {CLASSES[pred_test[i]]} ({p_test[i].max():.2f})", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 12. Robustness check: what if the clinic's X-ray machine is different?
This dataset has no age or sex labels (all children aged 1–5 from one hospital), so we can't do a demographic fairness split. Instead we test whether performance survives a **different-looking X-ray** (lower contrast, blur, lower resolution), which is a real issue when a tool moves to a new clinic. Say this limitation in your write-up.

In [ ]:
def shifted_tf(kind):
    ops = [T.Resize((IMG_SIZE, IMG_SIZE))]
    if kind == "low contrast": ops.append(T.ColorJitter(contrast=(0.5, 0.5)))
    if kind == "blur": ops.append(T.GaussianBlur(5, sigma=1.5))
    if kind == "low resolution": ops = [T.Resize((96, 96)), T.Resize((IMG_SIZE, IMG_SIZE))]
    ops.append(T.ToTensor()); return T.Compose(ops)

rob = []
for kind in ["original", "low contrast", "blur", "low resolution"]:
    dl = DataLoader(XRays(test_df, shifted_tf(kind) if kind != "original" else eval_tf), batch_size=BATCH, num_workers=nw)
    p, y = predict_probs(dl)
    r = report(kind, y, adjusted_pred(p, chosen)); rob.append(r)
pd.DataFrame(rob)[["model", "accuracy", "macro_recall", "pneumonia_recall(any type)"]].round(3)

## 13. Fix the weakness: train a second CNN on "different clinic" X-rays
Step 12 showed our CNN breaks on X-rays that look different (lower contrast, blur, low resolution). Our fix: train a **second CNN from scratch** where, during training, images randomly get those same kinds of changes. This teaches the model that the *disease*, not the image quality, is what matters.
The test set itself is never changed for training. We only use the distortions to *evaluate* robustness.

In [ ]:
def downscale(im):
    s = random.choice([64, 96, 128])
    return im.resize((s, s)).resize((IMG_SIZE, IMG_SIZE))

clinic_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.3, contrast=(0.4, 1.4)),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.4),
    T.RandomApply([T.Lambda(downscale)], p=0.3),
    T.ToTensor(),
])
torch.manual_seed(SEED + 1)
model2 = PneumoType().to(device)
train_dl2 = DataLoader(XRays(train_df, clinic_tf), batch_size=BATCH, shuffle=True, num_workers=nw)
train_model(model2, train_dl2, "best_robust.pt")

In [ ]:
def loader(d, tf): return DataLoader(XRays(d, tf), batch_size=BATCH, num_workers=nw)

rows = []
for kind in ["original", "low contrast", "blur", "low resolution"]:
    tf = eval_tf if kind == "original" else shifted_tf(kind)
    p1, y = predict_probs(loader(test_df, tf), model)
    p2, _ = predict_probs(loader(test_df, tf), model2)
    for name, p in [("CNN 1 (original)", p1), ("CNN 2 (clinic-robust)", p2), ("Ensemble of both", (p1 + p2) / 2)]:
        rows.append({"test images": kind, "model": name,
                     "accuracy": accuracy_score(y, p.argmax(1)),
                     "macro_recall": recall_score(y, p.argmax(1), average="macro")})
rob_df = pd.DataFrame(rows)
display(rob_df.pivot(index="test images", columns="model", values="macro_recall").round(3))
rob_df.pivot(index="test images", columns="model", values="macro_recall").plot.bar(figsize=(9, 4), ylim=(0, 1), rot=0)
plt.ylabel("macro recall (test)"); plt.title("Robustness: original vs clinic-robust CNN"); plt.axhline(1/K, ls="--", c="gray"); plt.show()

## 14. Pick the final model (on validation only)
Candidates: CNN 1, CNN 2, their **ensemble** (average of both), and the ensemble with **test-time augmentation (TTA)**, which also averages predictions on a slightly zoomed copy of each X-ray. We choose using **validation** macro recall, never test, then report the winner on test.

In [ ]:
tta_tf = T.Compose([T.Resize((248, 248)), T.CenterCrop(IMG_SIZE), T.ToTensor()])

def all_probs(d):
    p1, y = predict_probs(loader(d, eval_tf), model); p2, _ = predict_probs(loader(d, eval_tf), model2)
    t1, _ = predict_probs(loader(d, tta_tf), model);  t2, _ = predict_probs(loader(d, tta_tf), model2)
    return {"CNN 1": p1, "CNN 2 (robust)": p2, "Ensemble": (p1 + p2) / 2,
            "Ensemble + TTA": (p1 + p2 + t1 + t2) / 4}, y

val_c, y_val = all_probs(val_df)
test_c, y_test = all_probs(test_df)
sel = pd.DataFrame({"val macro recall": {k: recall_score(y_val, v.argmax(1), average="macro") for k, v in val_c.items()},
                    "test macro recall": {k: recall_score(y_test, v.argmax(1), average="macro") for k, v in test_c.items()}})
display(sel.round(3))
FINAL = sel["val macro recall"].idxmax()
print("Final model chosen on validation:", FINAL)
p_val_f, p_test_f = val_c[FINAL], test_c[FINAL]

## 15. Calibration: does "90% confident" really mean right 90% of the time?
The "Unsure, refer" mode only works if the confidence numbers are honest. We measure **expected calibration error (ECE)** and fix it with **temperature scaling** (one number, fitted on validation, that softens or sharpens the probabilities without changing which class wins).

In [ ]:
def ece(p, y, bins=10):
    conf, pred = p.max(1), p.argmax(1); e = 0
    edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e

def temp_scale(p, Tt):
    z = np.log(np.clip(p, 1e-8, 1)) / Tt
    z = np.exp(z - z.max(1, keepdims=True)); return z / z.sum(1, keepdims=True)

def nll(p, y): return -np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-8, 1)))

TEMP = min(np.arange(0.3, 4.01, 0.05), key=lambda t: nll(temp_scale(p_val_f, t), y_val))
p_val_cal, p_test_cal = temp_scale(p_val_f, TEMP), temp_scale(p_test_f, TEMP)
print(f"temperature = {TEMP:.2f} | test ECE before {ece(p_test_f, y_test):.3f} -> after {ece(p_test_cal, y_test):.3f}")

fig, ax = plt.subplots(figsize=(5, 5))
for name, p in [("before", p_test_f), ("after", p_test_cal)]:
    conf, cor = p.max(1), (p.argmax(1) == y_test)
    edges = np.linspace(0, 1, 11); xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 5: xs.append(conf[m].mean()); ys.append(cor[m].mean())
    ax.plot(xs, ys, "o-", label=f"{name} calibration")
ax.plot([0, 1], [0, 1], "--", c="gray", label="perfect"); ax.set_xlabel("model confidence"); ax.set_ylabel("actual accuracy")
ax.legend(); ax.set_title("Reliability diagram (test)"); plt.show()

## 16. Final model + safety tuning, with 95% confidence intervals
We redo the safety bonus for the final calibrated model (chosen on validation), then report test results with **95% confidence intervals** from a bootstrap that resamples **patients** (not images), because one child can have several X-rays.

In [ ]:
FINAL_BONUS = 0.0
for b in np.arange(0, 1.0, 0.01):
    if recall_score(y_val, adjusted_pred(p_val_cal, b), labels=[1], average=None)[0] >= TARGET:
        FINAL_BONUS = b; break
print(f"final {CLASSES[1]} bonus (validation): {FINAL_BONUS:.2f}")

pred_final = p_test_cal.argmax(1)
pred_final_safe = adjusted_pred(p_test_cal, FINAL_BONUS)
results.append(report(f"FINAL: {FINAL}", y_test, pred_final))
results.append(report(f"FINAL: {FINAL} + safety bonus {FINAL_BONUS:.2f}", y_test, pred_final_safe))
display(pd.DataFrame(results).round(3))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred_final, labels=list(range(K))), display_labels=CLASSES).plot(cmap="Blues")
plt.title(f"FINAL ({FINAL}) confusion matrix, test"); plt.show()

pat_idx = test_df.groupby("patient").indices; pats = list(pat_idx)
rng = np.random.default_rng(SEED); boot = []
for _ in range(1000):
    idx = np.concatenate([pat_idx[p] for p in rng.choice(pats, len(pats))])
    yt, yp = y_test[idx], pred_final[idx]
    boot.append([accuracy_score(yt, yp), recall_score(yt, yp, average="macro"),
                 recall_score((yt > 0).astype(int), (yp > 0).astype(int), zero_division=0)])
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
point = [accuracy_score(y_test, pred_final), recall_score(y_test, pred_final, average="macro"),
         recall_score((y_test > 0).astype(int), (pred_final > 0).astype(int))]
ci = pd.DataFrame({"value": point, "95% CI low": lo, "95% CI high": hi},
                  index=["accuracy", "macro recall", "pneumonia recall (any type)"])
display(ci.round(3))

In [ ]:
# Unsure band on the final calibrated model
unsure_rows = []
for cut in [0.5, 0.6, 0.7, 0.8, 0.9]:
    keep = p_test_cal.max(1) >= cut
    acc = accuracy_score(y_test[keep], pred_final_safe[keep]) if keep.any() else float("nan")
    unsure_rows.append({"confidence cutoff": cut, "share decided": keep.mean(), "accuracy on decided": acc, "referred to doctor": int((~keep).sum())})
unsure_df = pd.DataFrame(unsure_rows); display(unsure_df.round(3))

## 17. Shortcut test: is the model really looking at the lungs?
A famous failure of medical AI is "shortcut learning": a model scores well by using hospital markers, image borders or exposure instead of the disease. We test this directly: **black out the middle of the X-ray (the lungs)** and see how well the model can still "diagnose". If it does much better than chance (macro recall 0.33) with no lungs visible, part of its accuracy comes from shortcuts.

In [ ]:
def mask_center(t):
    t = t.clone(); h, w = t.shape[-2:]
    t[..., int(.12*h):int(.92*h), int(.15*w):int(.85*w)] = 0.0; return t
def mask_border(t):
    keep = t.clone(); out = torch.zeros_like(t); h, w = t.shape[-2:]
    sl = (..., slice(int(.12*h), int(.92*h)), slice(int(.15*w), int(.85*w)))
    out[sl] = keep[sl]; return out

sc_rows = []
for name, fn in [("full X-ray", None), ("lungs blacked out (border only)", mask_center), ("border blacked out (lungs only)", mask_border)]:
    tf = eval_tf if fn is None else T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Lambda(fn)])
    p1, y = predict_probs(loader(test_df, tf), model); p2, _ = predict_probs(loader(test_df, tf), model2)
    p = (p1 + p2) / 2
    sc_rows.append({"input": name, "accuracy": accuracy_score(y, p.argmax(1)), "macro_recall": recall_score(y, p.argmax(1), average="macro")})
shortcut_df = pd.DataFrame(sc_rows); display(shortcut_df.round(3))
print("chance macro recall =", round(1 / K, 3))

ex = Image.open(test_df.path[0]).convert("L")
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (title, fn) in zip(axes, [("full", None), ("lungs blacked out", mask_center), ("border blacked out", mask_border)]):
    t = eval_tf(ex); t = fn(t) if fn else t
    ax.imshow(t[0], cmap="gray"); ax.set_title(title); ax.axis("off")
plt.show()

## 18. Everyone, not just children: adults + children with **federated learning**
The children's dataset can only tell us about children. To cover **everyone**, we add adult X-rays from the **COVID-19 Radiography Database** (adult normal and lung-opacity/pneumonia X-rays from RSNA, plus adult COVID-19 pneumonia X-rays). No public adult dataset labels bacterial vs viral, so this part does **pneumonia vs normal** for everyone; the type (bacterial/viral) stays a children-only feature.

**Leakage check:** that database re-uses 1,341 "normal" X-rays and all its "viral pneumonia" X-rays from the same children's dataset we already use. We **remove** those using the database's own source metadata, so no image appears in both.

**Federated learning:** imagine the children's hospital and the adult hospital can't share patient X-rays (privacy law). With **FedAvg**, each hospital trains the shared model on its own X-rays for one round, sends back only the model **weights**, and a server averages them. The X-rays never leave the hospital. We compare 4 setups, all trained from scratch on the same budget:
1. **Children's hospital only**, 2. **Adult hospital only**, 3. **Pooled** (all X-rays in one place: best case but not private), 4. **Federated** (private).

In [ ]:
import copy
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

md_files = glob.glob(f"{DATA_ROOT}/**/Normal.metadata.xlsx", recursive=True)
ADULT_OK = len(md_files) > 0
print("Adult dataset found:", ADULT_OK)
if ADULT_OK:
    aroot = os.path.dirname(md_files[0])
    lookup = {os.path.splitext(os.path.basename(q))[0].lower(): q for q in glob.glob(f"{aroot}/**/*.png", recursive=True)
              if "/masks/" not in q.replace("\\\\", "/")}
    def load_cls(cls, label):
        m = pd.read_excel(f"{aroot}/{cls}.metadata.xlsx")
        m.columns = [str(c).strip().upper() for c in m.columns]
        d = pd.DataFrame({"path": m["FILE NAME"].astype(str).str.lower().map(lookup),
                          "url": m["URL"].astype(str), "source": cls, "y": label})
        return d.dropna(subset=["path"])
    a_norm = load_cls("Normal", 0)
    before = len(a_norm)
    a_norm = a_norm[~a_norm.url.str.contains("chest-xray-pneumonia|paultimothymooney", case=False)]
    print(f"removed {before - len(a_norm)} 'normal' images that came from the children's dataset")
    a_lo, a_cov = load_cls("Lung_Opacity", 1), load_cls("COVID", 1)
    adult_df = pd.concat([a_norm.sample(min(3000, len(a_norm)), random_state=SEED),
                          a_lo.sample(min(1500, len(a_lo)), random_state=SEED),
                          a_cov.sample(min(1500, len(a_cov)), random_state=SEED)]).reset_index(drop=True)
    adult_df["patient"] = adult_df.path
    a_tr, a_rest = train_test_split(adult_df, test_size=0.3, stratify=adult_df.y, random_state=SEED)
    a_va, a_te = train_test_split(a_rest, test_size=0.5, stratify=a_rest.y, random_state=SEED)
    print(adult_df.groupby(["source", "y"]).size())
    binarize = lambda d: d.assign(y=(d.y > 0).astype(int)).reset_index(drop=True)
    k_tr, k_va, k_te = binarize(train_df), binarize(val_df), binarize(test_df)
    a_tr, a_va, a_te = [d.reset_index(drop=True) for d in (a_tr, a_va, a_te)]

In [ ]:
FL_EPOCHS = 8
if ADULT_OK:
    def dl_(d, tf=eval_tf, shuffle=False): return DataLoader(XRays(d, tf), batch_size=BATCH, shuffle=shuffle, num_workers=nw)
    k_tr_dl, a_tr_dl = dl_(k_tr, train_tf, True), dl_(a_tr, train_tf, True)
    pool_tr_dl = dl_(pd.concat([k_tr, a_tr]), train_tf, True)
    both_va = pd.concat([k_va, a_va]).reset_index(drop=True)

    def bin_weight(d):
        c = np.bincount(d.y, minlength=2); return torch.tensor(c.sum() / (2 * c), dtype=torch.float32, device=device)

    def one_epoch(m, dl, w):
        crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
        scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda")); m.train()
        for x, y in dl:
            x, y = x.to(device), y.to(device); opt.zero_grad()
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                loss = crit(m(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()

    def bal_acc(m, d):
        p, y = predict_probs(dl_(d), m); return balanced_accuracy_score(y, p.argmax(1))

    def train_bin(name, dl, val_d):
        torch.manual_seed(SEED); m = PneumoType(2).to(device); w = bin_weight(dl.dataset.d); best, best_state = -1, None
        for ep in range(FL_EPOCHS):
            one_epoch(m, dl, w); b = bal_acc(m, val_d)
            if b > best: best, best_state = b, copy.deepcopy(m.state_dict())
            print(f"{name} epoch {ep+1}: val balanced acc {b:.3f}")
        m.load_state_dict(best_state); return m

    def fedavg(clients, val_d, rounds=FL_EPOCHS):
        torch.manual_seed(SEED); g = PneumoType(2).to(device); best, best_state = -1, None
        sizes = np.array([len(dl.dataset) for dl in clients], dtype=float); wts = sizes / sizes.sum()
        for r in range(rounds):
            states = []
            for dl in clients:                       # each hospital trains on its OWN X-rays
                local = copy.deepcopy(g); one_epoch(local, dl, bin_weight(dl.dataset.d)); states.append(local.state_dict())
            avg = {}
            for k_ in states[0]:                      # server only sees weights, never images
                if states[0][k_].is_floating_point(): avg[k_] = sum(w * s[k_] for w, s in zip(wts, states))
                else: avg[k_] = states[0][k_]
            g.load_state_dict(avg); b = bal_acc(g, val_d)
            if b > best: best, best_state = b, copy.deepcopy(g.state_dict())
            print(f"Federated round {r+1}: val balanced acc {b:.3f}")
        g.load_state_dict(best_state); return g

    m_kid = train_bin("Children only", k_tr_dl, k_va)
    m_adult = train_bin("Adults only", a_tr_dl, a_va)
    m_pool = train_bin("Pooled", pool_tr_dl, both_va)
    fed_model = fedavg([k_tr_dl, a_tr_dl], both_va)

In [ ]:
if ADULT_OK:
    def bin_metrics(m, d):
        p, y = predict_probs(dl_(d), m); pr = p.argmax(1)
        return {"balanced acc": balanced_accuracy_score(y, pr), "pneumonia caught (sensitivity)": recall_score(y, pr),
                "normal cleared (specificity)": recall_score(1 - y, 1 - pr), "AUC": roc_auc_score(y, p[:, 1])}
    fl_rows = []
    for name, m, shared in [("Children's hospital only", m_kid, "-"), ("Adult hospital only", m_adult, "-"),
                            ("Pooled (all X-rays shared)", m_pool, "YES"), ("Federated (FedAvg)", fed_model, "no, weights only")]:
        for pop, d in [("children", k_te), ("adults", a_te)]:
            fl_rows.append({"model": name, "X-rays shared?": shared, "tested on": pop, **bin_metrics(m, d)})
    fl_df = pd.DataFrame(fl_rows)
    display(fl_df.round(3))
    piv = fl_df.pivot(index="model", columns="tested on", values="balanced acc").loc[[r[0] for r in [("Children's hospital only",), ("Adult hospital only",), ("Pooled (all X-rays shared)",), ("Federated (FedAvg)",)]]]
    piv.plot.bar(figsize=(9, 4), ylim=(0.4, 1), rot=10); plt.axhline(0.5, ls="--", c="gray")
    plt.ylabel("balanced accuracy (test)"); plt.title("Pneumonia vs normal for everyone: who was the model trained on?"); plt.show()
    torch.save(fed_model.state_dict(), "fed_model.pt")

## 19. A new diagnosis: **tuberculosis**, and did the model learn the disease or the hospital?
TB kills more people than any other single infectious disease, and its X-rays can look like pneumonia. We add the **Tuberculosis (TB) Chest X-ray Database** (adults, TB and normal X-rays) as a **third hospital**, and train a system that sorts any X-ray into **normal / pneumonia / tuberculosis**, both pooled and with **federated learning across all 3 hospitals**.

**The danger:** each disease mostly comes from a different hospital. Pediatric pneumonia comes from one hospital, and every TB X-ray from another. A model could "cheat" by recognising *which hospital* an X-ray came from (machine, contrast, text markers) instead of the disease. We run **four checks**:
1. **Hospital detector:** train a CNN only to guess the hospital. If it's near-perfect, hospitals look different, so cheating is possible.
2. **Same-hospital test:** inside the TB hospital, normal and TB X-rays come from the same place, so the hospital can't explain the difference there.
3. **Normal X-rays from the TB hospital:** if the model calls them TB much more often than normals from other hospitals, it learned the hospital.
4. **Lungs blacked out:** if TB is still detected with no lungs visible, it's a shortcut.

We also **remove duplicate images across datasets** (some databases re-use the same normal X-rays), using an image fingerprint.

In [ ]:
tb_dirs = [d for d in glob.glob(f"{DATA_ROOT}/**/Tuberculosis", recursive=True) if os.path.isdir(d)]
TB_OK = ADULT_OK and len(tb_dirs) > 0
print("TB dataset found:", TB_OK)
if TB_OK:
    troot = os.path.dirname(tb_dirs[0])
    tb_pos = sorted(glob.glob(f"{troot}/Tuberculosis/*.png"))
    tb_neg = sorted(glob.glob(f"{troot}/Normal/*.png"))
    tb_df = pd.concat([pd.DataFrame({"path": tb_pos, "y": 2}),
                       pd.DataFrame({"path": tb_neg, "y": 0}).sample(min(1400, len(tb_neg)), random_state=SEED)]).reset_index(drop=True)

    def fingerprint(path):
        a = np.asarray(Image.open(path).convert("L").resize((16, 16)), dtype=np.float32)
        return ((a > a.mean()).astype(np.uint8)).tobytes()
    seen = {fingerprint(q) for q in pd.concat([adult_df.path, k_tr.path, k_va.path, k_te.path])}
    tb_df["fp"] = tb_df.path.map(fingerprint)
    dup = tb_df.fp.isin(seen) | tb_df.fp.duplicated()
    print(f"removed {int(dup.sum())} TB-hospital images that duplicate images in the other datasets (or each other)")
    tb_df = tb_df[~dup].drop(columns="fp").reset_index(drop=True); tb_df["patient"] = tb_df.path
    t_tr, t_rest = train_test_split(tb_df, test_size=0.3, stratify=tb_df.y, random_state=SEED)
    t_va, t_te = train_test_split(t_rest, test_size=0.5, stratify=t_rest.y, random_state=SEED)
    t_tr, t_va, t_te = [d.reset_index(drop=True) for d in (t_tr, t_va, t_te)]
    print("TB hospital:", tb_df.y.map({0: "normal", 2: "TB"}).value_counts().to_dict())
    # 3-way labels: 0 normal, 1 pneumonia, 2 TB. Hospital id: 0 children, 1 adult, 2 TB hospital
    H = {"children": (k_tr, k_va, k_te), "adults": (a_tr, a_va, a_te), "TB hospital": (t_tr, t_va, t_te)}
    for hid, (name, ds) in enumerate(H.items()):
        for d in ds: d["hospital"] = hid
    T3 = ["normal", "pneumonia", "tuberculosis"]

In [ ]:
if TB_OK:
    all_tr = pd.concat([v[0] for v in H.values()]).reset_index(drop=True)
    all_va = pd.concat([v[1] for v in H.values()]).reset_index(drop=True)
    all_te = pd.concat([v[2] for v in H.values()]).reset_index(drop=True)

    def w3(d):
        c = np.bincount(d.y, minlength=3).astype(float); c[c == 0] = 1
        return torch.tensor(c.sum() / (3 * c), dtype=torch.float32, device=device)
    def bal3(m, d):
        p, y = predict_probs(dl_(d), m); return balanced_accuracy_score(y, p.argmax(1))

    def train3(name, tr, va, epochs=FL_EPOCHS):
        torch.manual_seed(SEED); m = PneumoType(3).to(device); dl = dl_(tr, train_tf, True); best, bs = -1, None
        for ep in range(epochs):
            one_epoch(m, dl, w3(tr)); b = bal3(m, va)
            if b > best: best, bs = b, copy.deepcopy(m.state_dict())
            print(f"{name} epoch {ep+1}: val balanced acc {b:.3f}")
        m.load_state_dict(bs); return m

    def fed3(rounds=FL_EPOCHS):
        torch.manual_seed(SEED); g = PneumoType(3).to(device); best, bs = -1, None
        clients = [(dl_(v[0], train_tf, True), v[0]) for v in H.values()]
        wts = np.array([len(d) for _, d in clients], float); wts /= wts.sum()
        for r in range(rounds):
            states = []
            for dl, d in clients:
                local = copy.deepcopy(g); one_epoch(local, dl, w3(all_tr)); states.append(local.state_dict())
            g.load_state_dict({k_: (sum(w * s[k_] for w, s in zip(wts, states)) if states[0][k_].is_floating_point() else states[0][k_]) for k_ in states[0]})
            b = bal3(g, all_va)
            if b > best: best, bs = b, copy.deepcopy(g.state_dict())
            print(f"Federated (3 hospitals) round {r+1}: val balanced acc {b:.3f}")
        g.load_state_dict(bs); return g

    tb_pool = train3("Pooled 3-way", all_tr, all_va)
    tb_fed = fed3()

In [ ]:
if TB_OK:
    def eval3(m, d):
        p, y = predict_probs(dl_(d), m); pr = p.argmax(1); out = {"balanced acc": balanced_accuracy_score(y, pr)}
        for c in sorted(set(y)): out[f"recall {T3[c]}"] = recall_score(y == c, pr == c)
        return out
    rows = []
    for mname, m in [("Pooled (shares X-rays)", tb_pool), ("Federated, 3 hospitals (weights only)", tb_fed)]:
        rows.append({"model": mname, "tested on": "all hospitals", **eval3(m, all_te)})
        for hname, (_, _, te) in H.items(): rows.append({"model": mname, "tested on": hname, **eval3(m, te)})
    tb_res = pd.DataFrame(rows); display(tb_res.round(3))
    TB_MODEL = tb_pool if tb_res.query("model=='Pooled (shares X-rays)' and `tested on`=='all hospitals'")["balanced acc"].iloc[0] >= \
               tb_res.query("`tested on`=='all hospitals'")["balanced acc"].iloc[1] else tb_fed
    p, y = predict_probs(dl_(all_te), TB_MODEL)
    ConfusionMatrixDisplay(confusion_matrix(y, p.argmax(1), labels=[0, 1, 2]), display_labels=T3).plot(cmap="Purples")
    plt.title("Normal / pneumonia / TB: all 3 hospitals (test)"); plt.show()

In [ ]:
if TB_OK:
    # Check 1: can a CNN tell which hospital an X-ray came from?
    probe_tr = all_tr.assign(y=all_tr.hospital); probe_va = all_va.assign(y=all_va.hospital); probe_te = all_te.assign(y=all_te.hospital)
    probe = train3("Hospital detector", probe_tr, probe_va, epochs=4)
    pp, yy = predict_probs(dl_(probe_te), probe)
    hosp_acc = accuracy_score(yy, pp.argmax(1))
    print(f"CHECK 1: hospital detector accuracy = {hosp_acc:.3f} (chance = 0.333). High means hospitals look different, so shortcuts are POSSIBLE.")

    # Check 2: inside the TB hospital only (same source for normal and TB)
    p, y = predict_probs(dl_(t_te), TB_MODEL); pr = p.argmax(1)
    tb_sens = recall_score(y == 2, pr == 2); tb_spec = recall_score(y == 0, pr != 2)
    print(f"CHECK 2: within the TB hospital: TB caught {tb_sens:.3f}, normal X-rays NOT called TB {tb_spec:.3f}")

    # Check 3: normal X-rays from each hospital wrongly called TB
    fa = {}
    for hname, (_, _, te) in H.items():
        n = te[te.y == 0]
        if len(n): p, _ = predict_probs(dl_(n), TB_MODEL); fa[hname] = float((p.argmax(1) == 2).mean())
    print("CHECK 3: share of NORMAL X-rays called TB, by hospital:", {k_: round(v, 3) for k_, v in fa.items()})

    # Check 4: lungs blacked out
    lung_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Lambda(mask_center)])
    p, y = predict_probs(DataLoader(XRays(t_te, lung_tf), batch_size=BATCH, num_workers=nw), TB_MODEL)
    tb_sens_masked = recall_score(y == 2, p.argmax(1) == 2)
    print(f"CHECK 4: TB caught with LUNGS BLACKED OUT = {tb_sens_masked:.3f} (vs {tb_sens:.3f} with lungs). A big drop means it uses the lungs.")
    tb_checks = pd.DataFrame({"check": ["1. hospital detector accuracy (chance 0.33)", "2. TB caught inside TB hospital",
                                        "2. TB-hospital normals not called TB", "4. TB caught with lungs blacked out"]
                                       + [f"3. normals called TB: {k_}" for k_ in fa],
                              "value": [hosp_acc, tb_sens, tb_spec, tb_sens_masked] + list(fa.values())})
    display(tb_checks.round(3))

## 20. Heatmap: what is the model looking at? (Grad-CAM)
Red = regions that pushed the model toward its answer (from the clinic-robust CNN).

In [ ]:
def gradcam(img_pil, m=None, target=None):
    m = m or model2; m.eval()
    x = eval_tf(img_pil.convert("L")).unsqueeze(0).to(device)
    acts = {}
    def fwd_hook(mod, i, o):
        acts["a"] = o; o.register_hook(lambda g: acts.__setitem__("g", g))
    h = m.backbone.features.register_forward_hook(fwd_hook)
    logits = m(x); h.remove()
    cls = int(logits.argmax(1)) if target is None else target
    m.zero_grad(); logits[0, cls].backward()
    w = acts["g"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * acts["a"]).sum(1)).squeeze().detach().cpu().numpy()
    cam = np.array(Image.fromarray(cam).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

@torch.no_grad()
def final_probs_one(img_pil):
    xs = [eval_tf(img_pil.convert("L")).unsqueeze(0).to(device)]
    if "TTA" in FINAL: xs.append(tta_tf(img_pil.convert("L")).unsqueeze(0).to(device))
    ms = {"CNN 1": [model], "CNN 2 (robust)": [model2]}.get(FINAL, [model, model2])
    ps = [F.softmax(mm.eval()(x).float(), 1)[0].cpu().numpy() for mm in ms for x in xs]
    return temp_scale(np.mean(ps, 0, keepdims=True), TEMP)[0]

fig, axes = plt.subplots(1, K, figsize=(5 * K, 5))
for ax, c in zip(axes, range(K)):
    i = test_df.index[test_df.y == c][0]
    img = Image.open(test_df.path[i]).convert("L")
    pr = final_probs_one(img); cam = gradcam(img)
    ax.imshow(img.resize((IMG_SIZE, IMG_SIZE)), cmap="gray"); ax.imshow(cam, cmap="jet", alpha=0.35)
    ax.set_title(f"true {CLASSES[c]} | pred {CLASSES[int(pr.argmax())]} ({pr.max():.0%})"); ax.axis("off")
plt.show()

## 21. Demo app (Gradio)
Run this cell, click the public link it prints, then upload an X-ray or click one of the examples. Record your demo video from this.

In [ ]:
import gradio as gr, shutil, io
UNSURE_CUTOFF = 0.7   # see the unsure-band table in step 16

os.makedirs("examples", exist_ok=True); example_paths = []
for c in range(K):
    for j, i in enumerate(test_df.index[test_df.y == c][:2]):
        dst = f"examples/{CLASSES[c]}_{j}.jpeg"; shutil.copy(test_df.path[i], dst); example_paths.append(dst)

def fig_to_img(fig):
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=110, bbox_inches="tight"); plt.close(fig); buf.seek(0)
    return np.array(Image.open(buf).convert("RGB"))

# ---------- results dashboard (built once) ----------
def build_dashboard():
    n = 4 + int(ADULT_OK) + int(TB_OK); cols = 3; rows_ = int(np.ceil(n / cols))
    fig, axs = plt.subplots(rows_, cols, figsize=(17, 5 * rows_)); axs = axs.ravel()
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred_final, labels=list(range(K))), display_labels=CLASSES).plot(ax=axs[0], cmap="Blues", colorbar=False)
    axs[0].set_title("Final model: confusion matrix (children test set)")
    conf, cor = p_test_cal.max(1), (p_test_cal.argmax(1) == y_test); xs, ys = [], []
    for lo, hi in zip(np.linspace(0, 1, 11)[:-1], np.linspace(0, 1, 11)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 5: xs.append(conf[m].mean()); ys.append(cor[m].mean())
    axs[1].plot(xs, ys, "o-", label="PneumoType"); axs[1].plot([0, 1], [0, 1], "--", c="gray", label="perfect")
    axs[1].set_title(f"Calibration (ECE = {ece(p_test_cal, y_test):.3f})"); axs[1].set_xlabel("confidence"); axs[1].set_ylabel("accuracy"); axs[1].legend()
    rob_df.pivot(index="test images", columns="model", values="macro_recall").plot.bar(ax=axs[2], rot=0, ylim=(0.5, 0.95))
    axs[2].set_title("Robustness to different X-ray machines"); axs[2].set_ylabel("macro recall")
    axs[3].bar(shortcut_df.input.str.replace(" (", "\n(", regex=False), shortcut_df.macro_recall, color=["#2563eb", "#dc2626", "#93c5fd"])
    axs[3].axhline(1 / K, ls="--", c="gray"); axs[3].set_ylim(0, 1); axs[3].set_title("Shortcut test (lungs blacked out = chance)")
    k = 4
    if ADULT_OK:
        fl_df.pivot(index="model", columns="tested on", values="balanced acc").plot.bar(ax=axs[k], rot=12, ylim=(0.5, 1))
        axs[k].set_title("Children + adults: federated vs others"); k += 1
    if TB_OK:
        tb_res[tb_res["tested on"] != "all hospitals"].pivot(index="tested on", columns="model", values="balanced acc").plot.bar(ax=axs[k], rot=0, ylim=(0.3, 1))
        axs[k].set_title("Normal / pneumonia / TB by hospital"); k += 1
    for a in axs[k:]: a.axis("off")
    plt.tight_layout(); return fig_to_img(fig)
DASH = build_dashboard()

def metrics_md():
    r = pd.DataFrame(results).set_index("model")
    fin, safe = r.iloc[-2], r.iloc[-1]; base = r.loc["Baseline: logistic regression on 32x32 pixels"]
    md_ = f'''### Children: normal / bacterial / viral (held-out test, {len(y_test)} X-rays)
| | Simple baseline | **PneumoType** | **+ safety tuning** |
|---|---|---|---|
| Accuracy | {base.accuracy:.1%} | **{fin.accuracy:.1%}** | {safe.accuracy:.1%} |
| Macro recall | {base.macro_recall:.1%} | **{fin.macro_recall:.1%}** | {safe.macro_recall:.1%} |
| Pneumonia caught | {base['pneumonia_recall(any type)']:.1%} | {fin['pneumonia_recall(any type)']:.1%} | **{safe['pneumonia_recall(any type)']:.1%}** |
| Dangerous misses (bacterial) | {int(base.iloc[-1])} | {int(fin.iloc[-1])} | **{int(safe.iloc[-1])}** |

**95% confidence intervals (bootstrapped by patient):** accuracy {ci.loc['accuracy','value']:.1%} [{ci.loc['accuracy','95% CI low']:.1%}-{ci.loc['accuracy','95% CI high']:.1%}], pneumonia caught {ci.iloc[2,0]:.1%} [{ci.iloc[2,1]:.1%}-{ci.iloc[2,2]:.1%}]
**Calibration error (ECE):** {ece(p_test_cal, y_test):.3f} | **When >=90% confident:** right {unsure_df.iloc[-1]['accuracy on decided']:.1%} of the time
'''
    if ADULT_OK:
        f = fl_df[fl_df.model.str.startswith("Federated")]
        md_ += f"\n**Federated learning (children + adults, no X-rays shared):** balanced accuracy {f.iloc[0]['balanced acc']:.1%} on children, {f.iloc[1]['balanced acc']:.1%} on adults"
    if TB_OK:
        md_ += f"\n\n**Tuberculosis (3 hospitals):** TB caught inside the TB hospital {tb_sens:.1%}; with lungs blacked out {tb_sens_masked:.1%}; hospital-detector accuracy {hosp_acc:.1%}"
    return md_

# historical accuracy at a given confidence (from the test set)
TEST_CONF, TEST_COR = p_test_cal.max(1), (pred_final_safe == y_test)
def track_record(c):
    m = np.abs(TEST_CONF - c) <= 0.05
    return (TEST_COR[m].mean(), int(m.sum())) if m.sum() >= 10 else (float("nan"), int(m.sum()))

@torch.no_grad()
def model_probs(m, pil):
    return F.softmax(m.eval()(eval_tf(pil.convert("L")).unsqueeze(0).to(device)).float(), 1)[0].cpu().numpy()

def analyze(img):
    if img is None: return "### Upload or pick an X-ray", None, None, None, None, None
    t0 = time.time(); pil = Image.fromarray(img) if not isinstance(img, Image.Image) else img
    p1, p2 = model_probs(model, pil), model_probs(model2, pil)
    probs = final_probs_one(pil); adj = probs.copy(); adj[1] += FINAL_BONUS; cls = int(adj.argmax())
    conf = float(probs.max()); ent = float(-(probs * np.log(probs + 1e-9)).sum() / np.log(K))
    agree = int(p1.argmax()) == int(p2.argmax()); acc_hist, n_hist = track_record(conf)
    cam = gradcam(pil, target=cls)
    base = np.array(pil.convert("RGB").resize((IMG_SIZE, IMG_SIZE))) / 255.0
    overlay = ((0.65 * base + 0.35 * plt.cm.jet(cam)[..., :3]) * 255).astype(np.uint8)
    lung_focus = float(cam[int(.12*IMG_SIZE):int(.92*IMG_SIZE), int(.15*IMG_SIZE):int(.85*IMG_SIZE)].sum() / (cam.sum() + 1e-8))
    every = None
    if ADULT_OK:
        fp = model_probs(fed_model, pil); every = {"normal": float(fp[0]), "pneumonia": float(fp[1])}
    tbl = None
    if TB_OK:
        tp = model_probs(TB_MODEL, pil); tbl = {n_: float(v) for n_, v in zip(T3, tp)}
    unsure = conf < UNSURE_CUTOFF or not agree
    if unsure:
        head = "## UNSURE: refer to a doctor / radiologist"
        why = "the two networks disagree" if not agree else f"confidence {conf:.0%} is below {UNSURE_CUTOFF:.0%}"
        head += f"\n*Why:* {why}."
    else:
        head = {0: "## Normal", 1: "## Bacterial pneumonia: likely needs antibiotics, see a doctor now",
                2: "## Viral pneumonia: see a doctor"}[cls] if MODE == "3class" else ("## Normal" if cls == 0 else "## Pneumonia likely")
    if tbl and tbl["tuberculosis"] > 0.5: head += f"\n**Tuberculosis signal: {tbl['tuberculosis']:.0%}**: needs TB testing."
    track = f"On test X-rays where PneumoType was ~{conf:.0%} confident, it was right **{acc_hist:.0%}** of the time (n={n_hist})." if not np.isnan(acc_hist) else ""
    head += f"\n\n{track}\n\n*Prototype only. Not medical advice.*"
    stats = pd.DataFrame([
        ["CNN 1 says", f"{CLASSES[int(p1.argmax())]} ({p1.max():.0%})"],
        ["CNN 2 (clinic-robust) says", f"{CLASSES[int(p2.argmax())]} ({p2.max():.0%})"],
        ["Networks agree?", "yes" if agree else "NO: flagged as unsure"],
        ["Final ensemble (TTA, calibrated) top class", f"{CLASSES[int(probs.argmax())]} ({conf:.0%})"],
        ["Decision after safety tuning", CLASSES[cls]],
        ["Uncertainty (normalized entropy)", f"{ent:.2f}  (0 = certain, 1 = coin flip)"],
        ["Safety bonus applied to bacterial", f"+{FINAL_BONUS:.2f}"],
        ["Heatmap focus inside lung region", f"{lung_focus:.0%}"],
        ["Pneumonia check, everyone (federated)", f"{every['pneumonia']:.0%}" if every else "n/a"],
        ["Tuberculosis probability (3-hospital model)", f"{tbl['tuberculosis']:.0%}" if tbl else "n/a"],
        ["Analysis time", f"{(time.time() - t0) * 1000:.0f} ms"],
    ], columns=["Signal", "Value"])
    return head, {c: float(p) for c, p in zip(CLASSES, probs)}, every, tbl, overlay, stats

with gr.Blocks(title="PneumoType", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# PneumoType: chest X-ray pneumonia triage\nTwo CNNs trained **from scratch**, ensembled, calibrated and safety-tuned, plus federated models for adults and tuberculosis. *Prototype for a children's clinic, not medical advice.*")
    with gr.Tab("Diagnose"):
        with gr.Row():
            with gr.Column(scale=1):
                inp = gr.Image(label="Chest X-ray", type="numpy")
                btn = gr.Button("Analyze X-ray", variant="primary")
                gr.Examples(example_paths, inputs=inp, label="Example X-rays from the test set (click one)")
            with gr.Column(scale=2):
                verdict = gr.Markdown()
                with gr.Row():
                    type_lbl = gr.Label(label="Pneumonia type (children model, calibrated)", num_top_classes=3)
                    every_lbl = gr.Label(label="Pneumonia check for anyone (federated)")
                    tb_lbl = gr.Label(label="Normal / pneumonia / TB (3-hospital model)")
                with gr.Row():
                    cam_out = gr.Image(label="Grad-CAM: where the model looked")
                    stats_out = gr.Dataframe(label="Under the hood", wrap=True)
        btn.click(analyze, inp, [verdict, type_lbl, every_lbl, tb_lbl, cam_out, stats_out])
        inp.change(analyze, inp, [verdict, type_lbl, every_lbl, tb_lbl, cam_out, stats_out])
    with gr.Tab("How good is it?"):
        gr.Markdown(metrics_md())
        gr.Image(DASH, label="Results dashboard (all on held-out test data)")
    with gr.Tab("How it works"):
        gr.Markdown(f'''
1. **Two CNNs trained from scratch** (5 conv blocks, ~4.7M parameters each). CNN 2 is trained on blurred, low-contrast and low-resolution X-rays so it works on other clinics' machines.
2. **Ensemble + test-time augmentation**, chosen on validation data (never test).
3. **Calibration** (temperature {float(TEMP):.2f}) so confidence matches accuracy.
4. **Safety tuning**: a +{FINAL_BONUS:.2f} bonus to bacterial makes missed bacterial cases rarer.
5. **Unsure mode**: if confidence < {UNSURE_CUTOFF:.0%} or the two networks disagree, it refers to a doctor.
6. **Federated learning**: hospitals train locally and share only model weights (FedAvg), never X-rays.
7. **Shortcut checks**: lungs blacked out, plus a hospital detector, to make sure it learns disease, not hospitals.
''')
demo.launch(share=True)

## 22. Numbers for your write-up

In [ ]:
print("=== KEY NUMBERS ===")
print("Final model:", FINAL, "| temperature:", round(float(TEMP), 2), "| safety bonus:", round(float(FINAL_BONUS), 2))
print(pd.DataFrame(results).round(3).to_string())
print(); print(ci.round(3).to_string())
print(); print(sel.round(3).to_string())
print(); print(rob_df.pivot(index="test images", columns="model", values="macro_recall").round(3).to_string())
print(); print(unsure_df.round(3).to_string())
print(); print(shortcut_df.round(3).to_string())
print(); print("confusion matrix (final):"); print(confusion_matrix(y_test, pred_final, labels=list(range(K))))
print(f"ECE before {ece(p_test_f, y_test):.3f} after {ece(p_test_cal, y_test):.3f}")
if ADULT_OK: print(); print(fl_df.round(3).to_string())
if TB_OK: print(); print(tb_res.round(3).to_string()); print(); print(tb_checks.round(3).to_string())

**Credits to list:** Kermany et al. 2018 chest X-ray dataset (Kaggle: paultimothymooney/chest-xray-pneumonia, CC BY 4.0), COVID-19 Radiography Database and Tuberculosis (TB) Chest X-ray Database (Rahman, Chowdhury et al., Kaggle: tawsifurrahman), PyTorch, scikit-learn, Gradio, and any AI assistants you used. No pretrained models.